## **Evaluate Input Interventions**

This notebook evaluates the controlled input interventions created in `04_create_input_interventions.ipynb` using the trained TF-IDF logistic regression classifier.

The objective is to measure how model predictions change when specific information sources are systematically modified:

1. the supplied target identity,
2. explicit candidate mentions in the retrieved context posts, and
3. label-correlated lexical cues.

All intervention datasets were constructed from the held-out human-annotated test set. Any data-driven intervention rules were derived exclusively from the training data before being applied to the test set.

The TF-IDF model is loaded in its previously trained and frozen state. No model fitting, hyperparameter tuning, vocabulary adaptation, or intervention-specific training is performed in this notebook.

Predictions on each modified input are compared with predictions on the corresponding original test example. For label-preserving interventions, changes in classification performance are also evaluated against the human gold labels. Target swapping is treated separately because the original stance label is no longer a valid gold label after changing the target.

---

### **1. Setup**

Load the libraries and define the paths to the fixed human test set, the frozen TF-IDF model, and the previously generated intervention datasets.

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd


DATA_DIR = Path("../data/preprocessed")
INTERVENTION_DIR = Path("../data/interventions")
MODEL_DIR = Path("../models/tfidf_logreg")

HUMAN_TEST_PATH = DATA_DIR / "human_test.parquet"
MODEL_PATH = MODEL_DIR / "model.joblib"

In [2]:
intervention_paths = {
    "target_masked": (INTERVENTION_DIR / "human_test_target_masked.parquet"),
    "target_swapped": (INTERVENTION_DIR / "human_test_target_swapped.parquet"),
    "candidate_mentions_masked": (INTERVENTION_DIR / "human_test_candidate_mentions_masked.parquet"),
    "lexical_cues_top10_masked": (INTERVENTION_DIR / "human_test_lexical_cues_top10_masked.parquet"),
    "lexical_cues_top25_masked": (INTERVENTION_DIR / "human_test_lexical_cues_top25_masked.parquet")
}

for seed in range(1, 6):
    intervention_paths[f"candidate_mentions_control_seed{seed}"] = (INTERVENTION_DIR / f"human_test_candidate_mentions_control_seed{seed}.parquet")

    intervention_paths[f"lexical_cues_top10_control_seed{seed}"] = (INTERVENTION_DIR / f"human_test_lexical_cues_top10_control_seed{seed}.parquet")

    intervention_paths[f"lexical_cues_top25_control_seed{seed}"] = (INTERVENTION_DIR / f"human_test_lexical_cues_top25_control_seed{seed}.parquet")

---

### **Load the fixed test set and frozen model**

The original human-annotated test set provides the reference examples and gold labels. The previously saved TF-IDF pipeline is loaded without modification and will be used for all original and intervention inputs.

In [3]:
human_test = pd.read_parquet(HUMAN_TEST_PATH)
tfidf_logreg = joblib.load(MODEL_PATH)

print(f"Human test examples: {len(human_test):,}")
print(f"Columns: {human_test.columns.tolist()}")

Human test examples: 890
Columns: ['UserId', 'TargetEntity', 'StanceLabel', 'ContextPosts']


---

### **2. Audit the Frozen TF-IDF Model**

Before applying the model to the test interventions, inspect the loaded pipeline and verify the label mapping used during training.

The intervention placeholders are also checked against the frozen TF-IDF vocabulary. This ensures that masking removes the selected information without unintentionally introducing placeholder tokens that already have learned model weights.

In [4]:
vectorizer = tfidf_logreg.named_steps["tfidf"]
classifier = tfidf_logreg.named_steps["classifier"]

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Classifier classes: {classifier.classes_.tolist()}")
print(f"Lowercase: {vectorizer.lowercase}")
print(f"N-gram range: {vectorizer.ngram_range}")
print(f"Minimum document frequency: {vectorizer.min_df}")

Vocabulary size: 203,461
Classifier classes: [0, 1, 2]
Lowercase: True
N-gram range: (1, 2)
Minimum document frequency: 2


In [5]:
LABEL2ID = {
    "Against": 0,
    "Favor": 1,
    "Neither": 2,
}

ID2LABEL = {
    value: key
    for key, value in LABEL2ID.items()
}

LABEL_ORDER = [
    "Against",
    "Favor",
    "Neither",
]

In [6]:
assert classifier.classes_.tolist() == list(ID2LABEL.keys())

print(
    "Classifier label order:",
    [ID2LABEL[class_id] for class_id in classifier.classes_],
)

Classifier label order: ['Against', 'Favor', 'Neither']


---

### **Check intervention placeholders**

The intervention datasets use fixed placeholders to mark removed information. Because the TF-IDF model was trained before these interventions were created, these placeholders should ideally not correspond to features in the fitted vocabulary.

With the vectorizer's word-based tokenization, the relevant placeholder tokens are `candidate`, `removed`, and `target_removed`.

In [7]:
placeholder_tokens = ["candidate", "removed", "target_removed",]

placeholder_vocabulary_check = pd.DataFrame({
    "token": placeholder_tokens,
    "in_vocabulary": [
        token in vectorizer.vocabulary_
        for token in placeholder_tokens
    ]
})

placeholder_vocabulary_check

,token,in_vocabulary
0,candidate,True
1,removed,True
2,target_removed,False


In [9]:
placeholder = "[XQZ]"

features = vectorizer.build_analyzer()(placeholder)

print("TF-IDF analyzer:", features)
print(
    "Vocabulary overlap:",
    [
        feature
        for feature in features
        if feature in vectorizer.vocabulary_
    ],
)

TF-IDF analyzer: ['xqz']
Vocabulary overlap: []


In [10]:
from transformers import AutoTokenizer

roberta_tokenizer = AutoTokenizer.from_pretrained(
    "roberta-base"
)

placeholder = "[XQZ]"

tokens = roberta_tokenizer.tokenize(placeholder)
token_ids = roberta_tokenizer(
    placeholder,
    add_special_tokens=False,
)["input_ids"]

print("RoBERTa tokens:", tokens)
print("Number of tokens:", len(token_ids))
print("Token IDs:", token_ids)

/opt/homebrew/Caskroom/miniforge/base/envs/nlp-transformers/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RoBERTa tokens: ['[', 'X', 'Q', 'Z', ']']
Number of tokens: 5
Token IDs: [10975, 1000, 1864, 1301, 742]


In [11]:
placeholder = "<unk>"

print("RoBERTa tokens:", roberta_tokenizer.tokenize(placeholder))
print(
    "RoBERTa token IDs:",
    roberta_tokenizer(
        placeholder,
        add_special_tokens=False,
    )["input_ids"],
)

features = vectorizer.build_analyzer()(placeholder)

print("TF-IDF analyzer:", features)
print(
    "TF-IDF vocabulary overlap:",
    [
        feature
        for feature in features
        if feature in vectorizer.vocabulary_
    ],
)

RoBERTa tokens: ['<unk>']
RoBERTa token IDs: [3]
TF-IDF analyzer: ['unk']
TF-IDF vocabulary overlap: []


In [12]:
placeholder = "<unk>"

features = vectorizer.build_analyzer()(placeholder)

print("TF-IDF analyzer:", features)
print(
    "TF-IDF vocabulary overlap:",
    [
        feature
        for feature in features
        if feature in vectorizer.vocabulary_
    ],
)

TF-IDF analyzer: ['unk']
TF-IDF vocabulary overlap: []


In [13]:
placeholder = "redacted"
# RoBERTa
print(
    "RoBERTa tokens:",
    roberta_tokenizer.tokenize(
        f" {placeholder}"
    ),
)

print(
    "RoBERTa token IDs:",
    roberta_tokenizer(
        f" {placeholder}",
        add_special_tokens=False,
    )["input_ids"],
)

RoBERTa tokens: ['Ġredacted']
RoBERTa token IDs: [33706]


In [14]:
placeholder = "redacted"
# TF-IDF
features = vectorizer.build_analyzer()(placeholder)

print("TF-IDF analyzer:", features)
print(
    "TF-IDF vocabulary overlap:",
    [
        feature
        for feature in features
        if feature in vectorizer.vocabulary_
    ],
)


TF-IDF analyzer: ['redacted']
TF-IDF vocabulary overlap: ['redacted']


In [15]:
candidate_placeholders = [
    "placeholder",
    "maskeditem",
    "hiddenitem",
    "omitteditem",
    "concealeditem",
    "voidtoken",
    "blanktoken",
    "maskedtext",
]

rows = []

for placeholder in candidate_placeholders:
    tfidf_features = vectorizer.build_analyzer()(placeholder)
    tfidf_overlap = [
        feature
        for feature in tfidf_features
        if feature in vectorizer.vocabulary_
    ]

    roberta_tokens = roberta_tokenizer.tokenize(
        f" {placeholder}"
    )

    rows.append({
        "placeholder": placeholder,
        "tfidf_features": tfidf_features,
        "tfidf_overlap": tfidf_overlap,
        "roberta_tokens": roberta_tokens,
        "roberta_n_tokens": len(roberta_tokens),
    })

pd.DataFrame(rows)

,placeholder,tfidf_features,tfidf_overlap,roberta_tokens,roberta_n_tokens
0,placeholder,[placeholder],[],[Ġplaceholder],1
1,maskeditem,[maskeditem],[],"[Ġmasked, item]",2
2,hiddenitem,[hiddenitem],[],"[Ġhidden, item]",2
3,omitteditem,[omitteditem],[],"[Ġomitted, item]",2
4,concealeditem,[concealeditem],[],"[Ġconcealed, item]",2
5,voidtoken,[voidtoken],[],"[Ġvoid, token]",2
6,blanktoken,[blanktoken],[],"[Ġblank, token]",2
7,maskedtext,[maskedtext],[],"[Ġmasked, text]",2


In [16]:
def contains_placeholder(context_posts):
    return any(
        isinstance(post.get("Content"), str)
        and "placeholder" in post["Content"].lower()
        for post in context_posts
    )


print(
    "Test contexts containing 'placeholder':",
    human_test["ContextPosts"].apply(
        contains_placeholder
    ).sum(),
)

print(
    "Targets containing 'placeholder':",
    human_test["TargetEntity"].str.contains(
        "placeholder",
        case=False,
        regex=False,
    ).sum(),
)

Test contexts containing 'placeholder': 0
Targets containing 'placeholder': 0


In [17]:
roberta_examples = [
    "I support placeholder strongly.",
    "placeholder is terrible.",
    "anti-placeholder voters",
    "#placeholder",
    "placeholder2024",
]

for text in roberta_examples:
    print(text)
    print(roberta_tokenizer.tokenize(text))
    print()

I support placeholder strongly.
['I', 'Ġsupport', 'Ġplaceholder', 'Ġstrongly', '.']

placeholder is terrible.
['place', 'holder', 'Ġis', 'Ġterrible', '.']

anti-placeholder voters
['anti', '-', 'place', 'holder', 'Ġvoters']

#placeholder
['#', 'place', 'holder']

placeholder2024
['place', 'holder', '20', '24']



In [18]:
vocab = roberta_tokenizer.get_vocab()

candidates = []

for token in vocab:
    # RoBERTa uses Ġ to mark a preceding space
    if not token.startswith("Ġ"):
        continue

    word = token[1:]

    # Keep simple lowercase alphabetic words only
    if not word.isalpha():
        continue

    if not word.islower():
        continue

    if len(word) < 4:
        continue

    # Require both sentence-initial and whitespace-prefixed
    # versions to exist as single RoBERTa tokens
    if word not in vocab:
        continue

    # Must not be a learned TF-IDF feature
    if word in vectorizer.vocabulary_:
        continue

    # Confirm actual tokenizer behavior
    if len(roberta_tokenizer.tokenize(word)) != 1:
        continue

    if len(roberta_tokenizer.tokenize(f" {word}")) != 1:
        continue

    candidates.append(word)

print(f"Candidates found: {len(candidates)}")
print(candidates[:100])

Candidates found: 534
['const', 'lang', 'gener', 'manufact', 'bytes', 'anim', 'fare', 'anthrop', 'hist', 'stretched', 'apes', 'arte', 'pron', 'produ', 'maid', 'digit', 'operated', 'apolog', 'initialized', 'ware', 'comb', 'phen', 'debian', 'decl', 'licensed', 'bats', 'pard', 'node', 'toggle', 'antic', 'args', 'binding', 'reck', 'lime', 'repe', 'layout', 'parser', 'beaut', 'derived', 'brid', 'examination', 'disk', 'cube', 'mediated', 'usable', 'paced', 'resp', 'unsigned', 'interface', 'tele', 'python', 'imag', 'bool', 'tones', 'loader', 'ming', 'commun', 'contained', 'blade', 'recy', 'introdu', 'authent', 'wards', 'verb', 'econom', 'ende', 'javascript', 'container', 'plugin', 'spect', 'finals', 'mand', 'este', 'crafted', 'ants', 'stro', 'href', 'breeding', 'omin', 'vict', 'socket', 'marg', 'aston', 'oxide', 'width', 'diagn', 'timer', 'dict', 'gard', 'conf', 'rites', 'insured', 'comed', 'kins', 'requ', 'rend', 'param', 'endif', 'volt', 'cert']


In [19]:
candidate = "requ"

print("TF-IDF analyzer:",
      vectorizer.build_analyzer()(candidate))

print(
    "TF-IDF vocabulary overlap:",
    [
        feature
        for feature in vectorizer.build_analyzer()(candidate)
        if feature in vectorizer.vocabulary_
    ],
)

print(
    "RoBERTa initial:",
    roberta_tokenizer.tokenize(candidate)
)

print(
    "RoBERTa after space:",
    roberta_tokenizer.tokenize(f" {candidate}")
)

TF-IDF analyzer: ['requ']
TF-IDF vocabulary overlap: []
RoBERTa initial: ['requ']
RoBERTa after space: ['Ġrequ']


In [20]:
MASK_PLACEHOLDER = "requ"

def count_literal_occurrences(df, text_column, value):
    value = value.lower()

    count = 0

    for item in df[text_column]:
        if isinstance(item, str):
            count += value in item.lower()

        elif isinstance(item, list):
            for post in item:
                text = post.get("Content")
                if isinstance(text, str):
                    count += value in text.lower()

    return count

In [21]:
import re

pattern = re.compile(
    r"(?<!\w)requ(?!\w)",
    flags=re.IGNORECASE,
)

def count_placeholder_occurrences(df):
    count = 0

    for context_posts in df["ContextPosts"]:
        for post in context_posts:
            text = post.get("Content")

            if isinstance(text, str):
                count += len(pattern.findall(text))

    return count

In [24]:
import re

pattern = re.compile(
    r"(?<!\w)requ(?!\w)",
    flags=re.IGNORECASE,
)

def count_placeholder_occurrences(df):
    count = 0

    for context_posts in df["ContextPosts"]:
        for post in context_posts:
            text = post.get("Content")

            if isinstance(text, str):
                count += len(pattern.findall(text))

    return count

In [25]:
train = pd.read_parquet(
    DATA_DIR / "train.parquet"
)

print(
    "Training occurrences:",
    count_placeholder_occurrences(train),
)

print(
    "Test occurrences:",
    count_placeholder_occurrences(human_test),
)

Training occurrences: 0
Test occurrences: 0
